
# Silver Orders

## Objective

Create and validate the Silver Orders table from the Bronze Orders
table.

The Silver layer will standardize the order attributes, preserve the
source order records, create lifecycle quality flags, and enforce
pipeline-blocking quality checks.

---

## Source

`workspace.bronze.orders`

## Target

`workspace.silver.orders`

---

## Business Key

`order_id`

---

## Silver Responsibilities

- Preserve Bronze order records.
- Standardize order status.
- Convert order timestamps to timestamp datatypes.
- Create order lifecycle quality flags.
- Validate the order business key.
- Validate order lifecycle consistency.
- Reconcile Bronze and Silver row counts.
- Add `silver_load_timestamp`.

No Gold-layer metrics are created in this notebook.


## Step 1 — Import Required Libraries

In [0]:
from pyspark.sql import functions as F


## Step 2 — Read Bronze Orders

In [0]:
bronze_orders_df = spark.table(
    "workspace.bronze.orders"
)

bronze_order_count = bronze_orders_df.count()

print(
    f"Bronze Orders rows: {bronze_order_count:,}"
)

bronze_orders_df.printSchema()

Bronze Orders rows: 99,441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)




## Step 3 — Validate Bronze Orders Structure

In [0]:
required_order_columns = {
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
}

actual_order_columns = set(
    bronze_orders_df.columns
)

missing_order_columns = (
    required_order_columns
    - actual_order_columns
)

if missing_order_columns:
    raise ValueError(
        "Bronze Orders schema validation failed. "
        f"Missing columns: {sorted(missing_order_columns)}"
    )

print(
    "PASS — Bronze Orders contains all required columns."
)

PASS — Bronze Orders contains all required columns.



## Step 4 — Validate Bronze Order Business Key

In [0]:
null_order_ids = bronze_orders_df.filter(
    F.col("order_id").isNull()
).count()

duplicate_order_ids = (
    bronze_orders_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"NULL order IDs      : {null_order_ids}")
print(f"Duplicate order IDs : {duplicate_order_ids}")

if null_order_ids != 0:
    raise ValueError(
        "Bronze Orders quality gate failed: "
        "NULL order_id values found."
    )

if duplicate_order_ids != 0:
    raise ValueError(
        "Bronze Orders quality gate failed: "
        "Duplicate order_id values found."
    )

print("PASS — Bronze order business key is valid.")

NULL order IDs      : 0
Duplicate order IDs : 0
PASS — Bronze order business key is valid.



## Step 5 — Validate Order Lifecycle

The order lifecycle is evaluated in chronological order:

Purchase → Approval → Carrier Delivery → Customer Delivery.

Legitimate NULL timestamps are preserved because an order may not have
reached a particular lifecycle stage.

In [0]:
approval_before_purchase = bronze_orders_df.filter(
    F.col("order_approved_at").isNotNull()
    & (
        F.col("order_approved_at")
        < F.col("order_purchase_timestamp")
    )
).count()

carrier_before_approval = bronze_orders_df.filter(
    F.col("order_delivered_carrier_date").isNotNull()
    & F.col("order_approved_at").isNotNull()
    & (
        F.col("order_delivered_carrier_date")
        < F.col("order_approved_at")
    )
).count()

customer_delivery_before_carrier = bronze_orders_df.filter(
    F.col("order_delivered_customer_date").isNotNull()
    & F.col("order_delivered_carrier_date").isNotNull()
    & (
        F.col("order_delivered_customer_date")
        < F.col("order_delivered_carrier_date")
    )
).count()

print(
    f"Approval before purchase        : {approval_before_purchase}"
)

print(
    f"Carrier before approval         : {carrier_before_approval}"
)

print(
    f"Customer delivery before carrier: "
    f"{customer_delivery_before_carrier}"
)

Approval before purchase        : 0
Carrier before approval         : 1359
Customer delivery before carrier: 23



## Step 6 — Create Silver Orders DataFrame

Apply the required Silver transformations.

### Transformations

- Trim `order_id` and `customer_id`.
- Standardize `order_status` to lowercase.
- Convert all order lifecycle fields to TIMESTAMP.
- Create `approval_missing_flag`.
- Create `timeline_issue_flag`.
- Preserve all Bronze order records.
- Add `silver_load_timestamp`.

Lifecycle anomalies are retained and flagged rather than removed.

In [0]:
silver_orders_df = bronze_orders_df.select(
    F.trim(
        F.col("order_id")
    ).alias("order_id"),

    F.trim(
        F.col("customer_id")
    ).alias("customer_id"),

    F.lower(
        F.trim(
            F.col("order_status")
        )
    ).alias("order_status"),

    F.to_timestamp(
        F.col("order_purchase_timestamp")
    ).alias(
        "order_purchase_timestamp"
    ),

    F.to_timestamp(
        F.col("order_approved_at")
    ).alias(
        "order_approved_at"
    ),

    F.to_timestamp(
        F.col("order_delivered_carrier_date")
    ).alias(
        "order_delivered_carrier_date"
    ),

    F.to_timestamp(
        F.col("order_delivered_customer_date")
    ).alias(
        "order_delivered_customer_date"
    ),

    F.to_timestamp(
        F.col("order_estimated_delivery_date")
    ).alias(
        "order_estimated_delivery_date"
    ),

    (
        F.col("order_approved_at").isNull()
        & (
            F.lower(
                F.trim(
                    F.col("order_status")
                )
            ) == "delivered"
        )
    ).alias(
        "approval_missing_flag"
    ),

    (
        (
            F.col("order_approved_at").isNotNull()
            & F.col("order_purchase_timestamp").isNotNull()
            & (
                F.to_timestamp(
                    F.col("order_approved_at")
                )
                <
                F.to_timestamp(
                    F.col("order_purchase_timestamp")
                )
            )
        )
        |
        (
            F.col("order_delivered_carrier_date").isNotNull()
            & F.col("order_approved_at").isNotNull()
            & (
                F.to_timestamp(
                    F.col("order_delivered_carrier_date")
                )
                <
                F.to_timestamp(
                    F.col("order_approved_at")
                )
            )
        )
        |
        (
            F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_delivered_carrier_date").isNotNull()
            & (
                F.to_timestamp(
                    F.col("order_delivered_customer_date")
                )
                <
                F.to_timestamp(
                    F.col("order_delivered_carrier_date")
                )
            )
        )
    ).alias(
        "timeline_issue_flag"
    ),

    F.current_timestamp().alias(
        "silver_load_timestamp"
    )
)


## Preview Silver Orders

In [0]:
silver_orders_df.printSchema()

display(
    silver_orders_df.limit(20)
)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- approval_missing_flag: boolean (nullable = true)
 |-- timeline_issue_flag: boolean (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = false)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_missing_flag,timeline_issue_flag,silver_load_timestamp
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,false,false,2026-08-11T09:34:53.469Z



## Step 7 — Write Silver Orders

Persist the corrected Silver Orders DataFrame as a managed Delta table.

All Bronze order records are retained.

In [0]:
spark.sql("""
DROP TABLE IF EXISTS workspace.silver.orders
""")

(
    silver_orders_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.orders"
    )
)

print(
    "Silver Orders table created successfully."
)

Silver Orders table created successfully.



## Step 8 — Read Persisted Silver Orders

Reload the persisted Delta table so that all subsequent validations
operate on the actual Silver table.

In [0]:
silver_orders_df = spark.table(
    "workspace.silver.orders"
)

silver_order_count = silver_orders_df.count()

print(
    f"Silver Orders rows: "
    f"{silver_order_count:,}"
)

silver_orders_df.printSchema()

Silver Orders rows: 99,441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- approval_missing_flag: boolean (nullable = true)
 |-- timeline_issue_flag: boolean (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)




## Step 9 — Validate Bronze-to-Silver Row Count

Every Bronze order must be retained in Silver.

No order records are intentionally filtered during the Silver
transformation.

In [0]:
print(
    f"Bronze Orders rows : {bronze_order_count:,}"
)

print(
    f"Silver Orders rows : {silver_order_count:,}"
)

if bronze_order_count != silver_order_count:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "Bronze and Silver row counts do not match."
    )

print(
    "PASS — Bronze and Silver order counts match."
)

Bronze Orders rows : 99,441
Silver Orders rows : 99,441
PASS — Bronze and Silver order counts match.



## Step 10 — Validate Silver Order IDs

`order_id` is the business key for the Orders table.

It must be non-NULL and unique in the persisted Silver table.

In [0]:
silver_null_order_ids = silver_orders_df.filter(
    F.col("order_id").isNull()
).count()

silver_duplicate_order_ids = (
    silver_orders_df
    .groupBy("order_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    f"NULL order IDs      : "
    f"{silver_null_order_ids}"
)

print(
    f"Duplicate order IDs : "
    f"{silver_duplicate_order_ids}"
)

if silver_null_order_ids != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "NULL order_id values found."
    )

if silver_duplicate_order_ids != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "Duplicate order_id values found."
    )

print(
    "PASS — Silver order identifier validation passed."
)

NULL order IDs      : 0
Duplicate order IDs : 0
PASS — Silver order identifier validation passed.



## Step 11 — Validate Required Order Fields

The following fields are required for downstream order analytics and
relationships:

- `order_id`
- `customer_id`
- `order_status`
- `order_purchase_timestamp`

In [0]:
null_customer_ids = silver_orders_df.filter(
    F.col("customer_id").isNull()
).count()

null_order_statuses = silver_orders_df.filter(
    F.col("order_status").isNull()
).count()

null_purchase_timestamps = silver_orders_df.filter(
    F.col("order_purchase_timestamp").isNull()
).count()

print(
    f"NULL customer IDs        : "
    f"{null_customer_ids}"
)

print(
    f"NULL order statuses      : "
    f"{null_order_statuses}"
)

print(
    f"NULL purchase timestamps : "
    f"{null_purchase_timestamps}"
)

if null_customer_ids != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "NULL customer_id values found."
    )

if null_order_statuses != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "NULL order_status values found."
    )

if null_purchase_timestamps != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "NULL order_purchase_timestamp values found."
    )

print(
    "PASS — Required Silver Order fields are populated."
)

NULL customer IDs        : 0
NULL order statuses      : 0
NULL purchase timestamps : 0
PASS — Required Silver Order fields are populated.


In [0]:
expected_order_types = {
    "order_id": "string",
    "customer_id": "string",
    "order_status": "string",
    "order_purchase_timestamp": "timestamp",
    "order_approved_at": "timestamp",
    "order_delivered_carrier_date": "timestamp",
    "order_delivered_customer_date": "timestamp",
    "order_estimated_delivery_date": "timestamp",
    "approval_missing_flag": "boolean",
    "timeline_issue_flag": "boolean",
    "silver_load_timestamp": "timestamp"
}

actual_order_types = dict(
    silver_orders_df.dtypes
)

datatype_errors = {}

for column, expected_type in expected_order_types.items():
    actual_type = actual_order_types.get(column)

    if actual_type != expected_type:
        datatype_errors[column] = {
            "expected": expected_type,
            "actual": actual_type
        }

if datatype_errors:
    raise ValueError(
        "Silver Orders datatype validation failed: "
        f"{datatype_errors}"
    )

print(
    "PASS — Silver Orders datatypes are correct."
)

PASS — Silver Orders datatypes are correct.



## Step 12 — Validate Persisted Order Business Key

Validate the order identifier on the persisted Silver table.


## Step 13 — Validate Silver Load Timestamp

Every persisted Silver order must contain a Silver load timestamp for
pipeline traceability and operational monitoring.

In [0]:
null_silver_timestamps = silver_orders_df.filter(
    F.col("silver_load_timestamp").isNull()
).count()

print(
    f"NULL Silver load timestamps: "
    f"{null_silver_timestamps}"
)

if null_silver_timestamps != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "NULL silver_load_timestamp values found."
    )

print(
    "PASS — Silver load timestamp is populated."
)

NULL Silver load timestamps: 0
PASS — Silver load timestamp is populated.



## Step 14 — Validate Order Status Domain

Order status is standardized to lowercase in Silver.

Only known source order statuses are accepted.

In [0]:
valid_order_statuses = {
    "delivered",
    "shipped",
    "canceled",
    "unavailable",
    "invoiced",
    "processing",
    "created",
    "approved"
}

invalid_status_count = silver_orders_df.filter(
    F.col("order_status").isNull()
    |
    ~F.col("order_status").isin(
        list(valid_order_statuses)
    )
).count()

print(
    f"Invalid order statuses: "
    f"{invalid_status_count}"
)

if invalid_status_count != 0:
    display(
        silver_orders_df
        .filter(
            F.col("order_status").isNull()
            |
            ~F.col("order_status").isin(
                list(valid_order_statuses)
            )
        )
        .select(
            "order_id",
            "order_status"
        )
        .distinct()
    )

    raise ValueError(
        "Silver Orders quality gate failed: "
        "Invalid order_status values found."
    )

print(
    "PASS — All Silver order statuses are valid."
)

Invalid order statuses: 0
PASS — All Silver order statuses are valid.



## Step 15 — Validate Approval Missing Flag

A delivered order without an approval timestamp is flagged as a data
quality issue.

The order itself is retained.

In [0]:
expected_approval_missing_flag = (
    F.col("order_status") == "delivered"
) & F.col("order_approved_at").isNull()

incorrect_approval_flags = silver_orders_df.filter(
    F.col("approval_missing_flag")
    != expected_approval_missing_flag
).count()

approval_missing_count = silver_orders_df.filter(
    F.col("approval_missing_flag")
).count()

print(
    f"Orders with approval missing flag: "
    f"{approval_missing_count}"
)

print(
    f"Incorrect approval flags        : "
    f"{incorrect_approval_flags}"
)

if incorrect_approval_flags != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "approval_missing_flag does not match "
        "the defined business rule."
    )

print(
    "PASS — Approval missing flag is logically correct."
)

Orders with approval missing flag: 14
Incorrect approval flags        : 0
PASS — Approval missing flag is logically correct.



## Step 16 — Analyze Order Lifecycle Rules

The expected lifecycle is:

Purchase → Approval → Carrier Delivery → Customer Delivery

Each rule is evaluated independently.

Records violating a rule are retained and identified through the
existing `timeline_issue_flag`.

In [0]:
# ================================================================
# SILVER ORDERS — TIMELINE ANOMALY INSPECTION
# ================================================================

from pyspark.sql import functions as F

# ---------------------------------------------------------------
# 1. Count timeline anomalies
# ---------------------------------------------------------------

approval_before_purchase_count = (
    silver_orders_df
    .filter(
        F.col("order_approved_at").isNotNull()
        & F.col("order_purchase_timestamp").isNotNull()
        & (
            F.col("order_approved_at")
            < F.col("order_purchase_timestamp")
        )
    )
    .count()
)

carrier_before_approval_count = (
    silver_orders_df
    .filter(
        F.col("order_delivered_carrier_date").isNotNull()
        & F.col("order_approved_at").isNotNull()
        & (
            F.col("order_delivered_carrier_date")
            < F.col("order_approved_at")
        )
    )
    .count()
)

delivery_before_carrier_count = (
    silver_orders_df
    .filter(
        F.col("order_delivered_customer_date").isNotNull()
        & F.col("order_delivered_carrier_date").isNotNull()
        & (
            F.col("order_delivered_customer_date")
            < F.col("order_delivered_carrier_date")
        )
    )
    .count()
)

print(
    f"Approval before purchase        : "
    f"{approval_before_purchase_count:,}"
)

print(
    f"Carrier before approval         : "
    f"{carrier_before_approval_count:,}"
)

print(
    f"Customer delivery before carrier: "
    f"{delivery_before_carrier_count:,}"
)


# ---------------------------------------------------------------
# 2. Inspect carrier-before-approval records
# ---------------------------------------------------------------

carrier_before_approval_df = (
    silver_orders_df
    .filter(
        F.col("order_delivered_carrier_date").isNotNull()
        & F.col("order_approved_at").isNotNull()
        & (
            F.col("order_delivered_carrier_date")
            < F.col("order_approved_at")
        )
    )
)

print(
    "Carrier-before-approval records: "
    f"{carrier_before_approval_df.count():,}"
)

display(
    carrier_before_approval_df
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "timeline_issue_flag"
    )
    .orderBy(
        "order_delivered_carrier_date"
    )
)

Approval before purchase        : 0
Carrier before approval         : 1,359
Customer delivery before carrier: 23
Carrier-before-approval records: 1,359


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,timeline_issue_flag
6b80bb20190715d71c43efff617bd659,2fedecfd993b8b3fa889d00eee230748,delivered,2017-02-19T01:15:03.000Z,2017-03-01T10:51:46.000Z,2017-02-22T16:05:29.000Z,2017-02-24T14:27:26.000Z,2017-03-17T00:00:00.000Z,true
69a236fbbc4a603ebfa4468a3bdcb140,1bd6b1b425cc25c6e0c79f68d28fe2fb,delivered,2017-04-25T01:46:02.000Z,2017-04-27T10:32:00.000Z,2017-04-26T09:11:44.000Z,2017-05-03T13:39:47.000Z,2017-05-25T00:00:00.000Z,true
cc460ac4435835dfdda5526dfe84f01f,99a32bf8f0c54702217b584a4d220761,delivered,2017-04-25T10:10:12.000Z,2017-04-27T10:32:44.000Z,2017-04-26T09:16:59.000Z,2017-04-27T22:23:09.000Z,2017-05-12T00:00:00.000Z,true
104cf29a266be8f412ca30ec4cdab145,394c4558e06b0c9e8d6673e0d1c75d62,delivered,2017-04-25T22:50:48.000Z,2017-04-27T13:36:40.000Z,2017-04-26T09:47:10.000Z,2017-05-08T07:46:56.000Z,2017-05-12T00:00:00.000Z,true
2d0d4075ded592212bcd5e5bc561b406,ad4fdd8ba1535077790107fc697fbe97,delivered,2017-04-26T12:22:32.000Z,2017-04-26T13:06:25.000Z,2017-04-26T13:01:25.000Z,2017-05-15T07:38:03.000Z,2017-05-25T00:00:00.000Z,true
c85ea30e9a24abecb353caa26294c81c,f6b48a3dd3d76b9f3a58d8e108df430b,delivered,2017-04-26T08:19:14.000Z,2017-04-27T13:36:17.000Z,2017-04-26T14:52:01.000Z,2017-05-08T16:07:01.000Z,2017-05-23T00:00:00.000Z,true
af2adc7e31b52bdfe068cf60426b54b2,afd3f4f110c36e0ec5b412f4d30af5fc,delivered,2017-04-24T09:14:06.000Z,2017-04-27T10:32:39.000Z,2017-04-26T18:47:58.000Z,2017-05-02T17:24:56.000Z,2017-05-11T00:00:00.000Z,true
5a8a26a982045e5cd9d257b5fdef75c9,77f128633c92a6243ea55349236b8c23,delivered,2017-04-25T22:17:39.000Z,2017-04-27T13:33:28.000Z,2017-04-26T21:33:39.000Z,2017-05-12T12:17:48.000Z,2017-05-17T00:00:00.000Z,true
a60d571514c73cfc7655d67c75ed82c7,ad5945bdec9120fbc7eab1a9746695a7,delivered,2017-04-25T09:50:01.000Z,2017-04-27T10:32:03.000Z,2017-04-27T07:18:06.000Z,2017-05-05T18:58:07.000Z,2017-05-26T00:00:00.000Z,true
294a3f0960f3588f7b1597da079f590e,8ea42c1486fc37fb10dfb10210ea1e5a,delivered,2017-04-25T22:15:42.000Z,2017-04-27T13:33:23.000Z,2017-04-27T07:18:06.000Z,2017-05-08T13:13:20.000Z,2017-05-16T00:00:00.000Z,true



## Step 17 — Investigate Carrier-Before-Approval Records

Inspect orders where the carrier delivery timestamp occurs before the
approval timestamp.

These records are retained in Silver and identified through
`timeline_issue_flag`.

This investigation is performed to understand the source-data quality
issue without removing valid business records.

In [0]:
carrier_before_approval_df = silver_orders_df.filter(
    (F.col("order_delivered_carrier_date").isNotNull()) &
    (F.col("order_approved_at").isNotNull()) &
    (
        F.col("order_delivered_carrier_date")
        < F.col("order_approved_at")
    )
)

carrier_before_approval_count = (
    carrier_before_approval_df.count()
)

print(
    f"Carrier-before-approval records: "
    f"{carrier_before_approval_count:,}"
)

display(
    carrier_before_approval_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "timeline_issue_flag"
    ).orderBy(
        "order_delivered_carrier_date"
    )
)

Carrier-before-approval records: 1,359


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,timeline_issue_flag
6b80bb20190715d71c43efff617bd659,2fedecfd993b8b3fa889d00eee230748,delivered,2017-02-19T01:15:03.000Z,2017-03-01T10:51:46.000Z,2017-02-22T16:05:29.000Z,2017-02-24T14:27:26.000Z,2017-03-17T00:00:00.000Z,true
69a236fbbc4a603ebfa4468a3bdcb140,1bd6b1b425cc25c6e0c79f68d28fe2fb,delivered,2017-04-25T01:46:02.000Z,2017-04-27T10:32:00.000Z,2017-04-26T09:11:44.000Z,2017-05-03T13:39:47.000Z,2017-05-25T00:00:00.000Z,true
cc460ac4435835dfdda5526dfe84f01f,99a32bf8f0c54702217b584a4d220761,delivered,2017-04-25T10:10:12.000Z,2017-04-27T10:32:44.000Z,2017-04-26T09:16:59.000Z,2017-04-27T22:23:09.000Z,2017-05-12T00:00:00.000Z,true
104cf29a266be8f412ca30ec4cdab145,394c4558e06b0c9e8d6673e0d1c75d62,delivered,2017-04-25T22:50:48.000Z,2017-04-27T13:36:40.000Z,2017-04-26T09:47:10.000Z,2017-05-08T07:46:56.000Z,2017-05-12T00:00:00.000Z,true
2d0d4075ded592212bcd5e5bc561b406,ad4fdd8ba1535077790107fc697fbe97,delivered,2017-04-26T12:22:32.000Z,2017-04-26T13:06:25.000Z,2017-04-26T13:01:25.000Z,2017-05-15T07:38:03.000Z,2017-05-25T00:00:00.000Z,true
c85ea30e9a24abecb353caa26294c81c,f6b48a3dd3d76b9f3a58d8e108df430b,delivered,2017-04-26T08:19:14.000Z,2017-04-27T13:36:17.000Z,2017-04-26T14:52:01.000Z,2017-05-08T16:07:01.000Z,2017-05-23T00:00:00.000Z,true
af2adc7e31b52bdfe068cf60426b54b2,afd3f4f110c36e0ec5b412f4d30af5fc,delivered,2017-04-24T09:14:06.000Z,2017-04-27T10:32:39.000Z,2017-04-26T18:47:58.000Z,2017-05-02T17:24:56.000Z,2017-05-11T00:00:00.000Z,true
5a8a26a982045e5cd9d257b5fdef75c9,77f128633c92a6243ea55349236b8c23,delivered,2017-04-25T22:17:39.000Z,2017-04-27T13:33:28.000Z,2017-04-26T21:33:39.000Z,2017-05-12T12:17:48.000Z,2017-05-17T00:00:00.000Z,true
a60d571514c73cfc7655d67c75ed82c7,ad5945bdec9120fbc7eab1a9746695a7,delivered,2017-04-25T09:50:01.000Z,2017-04-27T10:32:03.000Z,2017-04-27T07:18:06.000Z,2017-05-05T18:58:07.000Z,2017-05-26T00:00:00.000Z,true
294a3f0960f3588f7b1597da079f590e,8ea42c1486fc37fb10dfb10210ea1e5a,delivered,2017-04-25T22:15:42.000Z,2017-04-27T13:33:23.000Z,2017-04-27T07:18:06.000Z,2017-05-08T13:13:20.000Z,2017-05-16T00:00:00.000Z,true



## Step 18 — Investigate Customer-Delivery-Before-Carrier Records

Inspect orders where customer delivery occurs before carrier delivery.

These records are retained in Silver and identified through
`timeline_issue_flag`.

In [0]:
delivery_before_carrier_df = silver_orders_df.filter(
    (F.col("order_delivered_customer_date").isNotNull()) &
    (F.col("order_delivered_carrier_date").isNotNull()) &
    (
        F.col("order_delivered_customer_date")
        < F.col("order_delivered_carrier_date")
    )
)

delivery_before_carrier_count = (
    delivery_before_carrier_df.count()
)

print(
    f"Customer-delivery-before-carrier records: "
    f"{delivery_before_carrier_count:,}"
)

display(
    delivery_before_carrier_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "timeline_issue_flag"
    ).orderBy(
        "order_delivered_customer_date"
    )
)

Customer-delivery-before-carrier records: 23


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,timeline_issue_flag
76458889992169d3135b264dc13aec67,999196dca58a3db3d966d8f148532010,delivered,2016-10-07T10:05:16.000Z,2016-10-07T11:24:43.000Z,2016-10-26T11:43:06.000Z,2016-10-20T18:03:17.000Z,2016-11-29T00:00:00.000Z,true
19feb5627c41ea1b36a8e50a469b3644,b8097c8f0c1f58ab56a53812a446a898,delivered,2016-10-07T17:09:56.000Z,2016-10-07T17:32:09.000Z,2016-10-26T11:42:05.000Z,2016-10-20T19:07:54.000Z,2016-12-01T00:00:00.000Z,true
f688669f48063536e082bb32d634cd46,691d28310063e5b36b732b117f2bcfc0,delivered,2016-10-07T10:28:56.000Z,2016-10-11T04:56:13.000Z,2016-10-21T18:02:40.000Z,2016-10-20T20:33:27.000Z,2016-11-29T00:00:00.000Z,true
8c78d01de3a9009e23d6877a7cc9be20,6cd7106899e59a1fbd0622d5f1efedf4,delivered,2016-10-08T15:36:50.000Z,2016-10-08T18:13:44.000Z,2016-10-26T11:41:53.000Z,2016-10-25T17:51:46.000Z,2016-11-30T00:00:00.000Z,true
d5558a097766363b8e76b38c43332e8a,49a4fe701c3d30eac6c180908e665ebb,delivered,2017-02-04T19:01:33.000Z,2017-02-04T19:31:12.000Z,2017-02-15T08:55:26.000Z,2017-02-10T07:58:32.000Z,2017-03-01T00:00:00.000Z,true
c1e2bf2b7dd3309f2f5356c6b63968fa,e37d47e7eec62f08dc5deecc7d5532d6,delivered,2017-02-10T10:19:10.000Z,2017-02-10T10:30:13.000Z,2017-03-02T17:34:26.000Z,2017-02-14T15:15:57.000Z,2017-03-15T00:00:00.000Z,true
b866af202be0692766081310cd4085e1,d1800078046ed2e5ae1b0792b695c56e,delivered,2017-01-27T14:59:17.000Z,2017-01-27T15:30:46.000Z,2017-02-20T02:32:08.000Z,2017-02-15T03:53:46.000Z,2017-04-17T00:00:00.000Z,true
771c4f1f521f462e4b95619e648aaeab,d424b3ce4c850247e8c84ff8752de868,delivered,2017-03-22T20:15:15.000Z,2017-03-22T20:15:15.000Z,2017-03-30T13:14:34.000Z,2017-03-28T17:28:32.000Z,2017-04-12T00:00:00.000Z,true
29941903985f944b0ffc49c479c1547d,b56ee98181afc3948a758d73a08423de,delivered,2017-05-29T16:16:50.000Z,2017-05-29T16:25:16.000Z,2017-06-09T15:07:29.000Z,2017-06-02T11:09:16.000Z,2017-06-23T00:00:00.000Z,true
b27af682321527a6349f1761eb3f360c,9859dd92e872dbaa60ca3cd5f0d7ad07,delivered,2017-06-14T20:17:04.000Z,2017-06-14T20:30:08.000Z,2017-06-27T14:51:54.000Z,2017-06-26T15:45:35.000Z,2017-07-14T00:00:00.000Z,true



## Step 19 — Validate Timeline Quality Flag

The `timeline_issue_flag` must be TRUE whenever an order violates
any defined lifecycle rule:

- Approval before purchase
- Carrier delivery before approval
- Customer delivery before carrier delivery

The flag must be FALSE when none of these rules are violated.

Lifecycle anomalies are retained in Silver and are not removed.

In [0]:
expected_timeline_issue = (
    (
        F.col("order_approved_at").isNotNull()
        & F.col("order_purchase_timestamp").isNotNull()
        & (
            F.col("order_approved_at")
            < F.col("order_purchase_timestamp")
        )
    )
    |
    (
        F.col("order_delivered_carrier_date").isNotNull()
        & F.col("order_approved_at").isNotNull()
        & (
            F.col("order_delivered_carrier_date")
            < F.col("order_approved_at")
        )
    )
    |
    (
        F.col("order_delivered_customer_date").isNotNull()
        & F.col("order_delivered_carrier_date").isNotNull()
        & (
            F.col("order_delivered_customer_date")
            < F.col("order_delivered_carrier_date")
        )
    )
)

incorrect_timeline_flags = silver_orders_df.filter(
    F.col("timeline_issue_flag")
    != expected_timeline_issue
).count()

timeline_issue_count = silver_orders_df.filter(
    F.col("timeline_issue_flag") == True
).count()

print(
    f"Orders with timeline issues : "
    f"{timeline_issue_count:,}"
)

print(
    f"Incorrect timeline flags    : "
    f"{incorrect_timeline_flags}"
)

if incorrect_timeline_flags != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "timeline_issue_flag does not match "
        "the defined lifecycle rules."
    )

print(
    "PASS — Timeline quality flag is logically correct."
)

Orders with timeline issues : 1,382
Incorrect timeline flags    : 0
PASS — Timeline quality flag is logically correct.



## Step 20 — Validate Bronze-to-Silver Order Population

Verify that the exact set of Bronze `order_id` values is preserved in
Silver.

This prevents a row-count match from hiding simultaneous record loss
and record creation during the Silver transformation.

In [0]:
bronze_order_ids_df = bronze_orders_df.select(
    "order_id"
).distinct()

silver_order_ids_df = silver_orders_df.select(
    "order_id"
).distinct()


bronze_ids_missing_in_silver = (
    bronze_order_ids_df
    .join(
        silver_order_ids_df,
        on="order_id",
        how="left_anti"
    )
    .count()
)


silver_ids_not_in_bronze = (
    silver_order_ids_df
    .join(
        bronze_order_ids_df,
        on="order_id",
        how="left_anti"
    )
    .count()
)


print(
    f"Bronze order IDs missing in Silver : "
    f"{bronze_ids_missing_in_silver}"
)

print(
    f"Silver order IDs not in Bronze     : "
    f"{silver_ids_not_in_bronze}"
)


if bronze_ids_missing_in_silver != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "Bronze order IDs are missing from Silver."
    )


if silver_ids_not_in_bronze != 0:
    raise ValueError(
        "Silver Orders quality gate failed: "
        "Silver contains order IDs not present in Bronze."
    )


print(
    "PASS — Bronze and Silver order populations match exactly."
)

Bronze order IDs missing in Silver : 0
Silver order IDs not in Bronze     : 0
PASS — Bronze and Silver order populations match exactly.



## Step 21 — Validate Timestamp Conversion

Verify that the Bronze-to-Silver timestamp conversion preserves the
expected NULL pattern.

A NULL timestamp in Silver is valid when the corresponding Bronze
timestamp was already NULL or blank.

Unexpected timestamp conversion failures must fail the pipeline.

In [0]:
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

timestamp_conversion_errors = {}

for column in timestamp_columns:

    bronze_expected_null_count = bronze_orders_df.filter(
        F.col(column).isNull()
        | (F.trim(F.col(column)) == "")
    ).count()

    silver_null_count = silver_orders_df.filter(
        F.col(column).isNull()
    ).count()

    if silver_null_count != bronze_expected_null_count:
        timestamp_conversion_errors[column] = {
            "Bronze expected NULL": bronze_expected_null_count,
            "Silver actual NULL": silver_null_count
        }


if timestamp_conversion_errors:

    raise ValueError(
        "Silver Orders timestamp conversion validation failed: "
        f"{timestamp_conversion_errors}"
    )


print(
    "PASS — Timestamp conversion preserved the expected NULL pattern."
)

PASS — Timestamp conversion preserved the expected NULL pattern.



## Step 22 — Final Silver Orders Quality Gate

Validate the persisted Silver Orders Delta table against all critical
structural, business, transformation, and operational quality rules.

Lifecycle anomalies are treated as data-quality observations and do not
fail the pipeline when they are correctly identified by
`timeline_issue_flag`.

In [0]:
final_silver_orders_df = spark.table(
    "workspace.silver.orders"
)

final_silver_order_count = (
    final_silver_orders_df.count()
)

final_null_order_ids = (
    final_silver_orders_df
    .filter(
        F.col("order_id").isNull()
    )
    .count()
)

final_duplicate_order_ids = (
    final_silver_orders_df
    .groupBy("order_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

final_null_customer_ids = (
    final_silver_orders_df
    .filter(
        F.col("customer_id").isNull()
    )
    .count()
)

final_null_statuses = (
    final_silver_orders_df
    .filter(
        F.col("order_status").isNull()
    )
    .count()
)

final_invalid_statuses = (
    final_silver_orders_df
    .filter(
        ~F.col("order_status").isin(
            list(valid_order_statuses)
        )
    )
    .count()
)

final_null_purchase_timestamps = (
    final_silver_orders_df
    .filter(
        F.col("order_purchase_timestamp").isNull()
    )
    .count()
)

final_null_silver_timestamps = (
    final_silver_orders_df
    .filter(
        F.col("silver_load_timestamp").isNull()
    )
    .count()
)

final_incorrect_approval_flags = (
    final_silver_orders_df
    .filter(
        F.col("approval_missing_flag")
        != expected_approval_missing_flag
    )
    .count()
)

final_incorrect_timeline_flags = (
    final_silver_orders_df
    .filter(
        F.col("timeline_issue_flag")
        != expected_timeline_issue
    )
    .count()
)

print("=" * 75)
print("SILVER ORDERS QUALITY GATE")
print("=" * 75)

print(
    f"Bronze rows                    : "
    f"{bronze_order_count:,}"
)

print(
    f"Silver rows                    : "
    f"{final_silver_order_count:,}"
)

print(
    f"NULL order IDs                 : "
    f"{final_null_order_ids}"
)

print(
    f"Duplicate order IDs            : "
    f"{final_duplicate_order_ids}"
)

print(
    f"NULL customer IDs              : "
    f"{final_null_customer_ids}"
)

print(
    f"NULL order statuses            : "
    f"{final_null_statuses}"
)

print(
    f"Invalid order statuses         : "
    f"{final_invalid_statuses}"
)

print(
    f"NULL purchase timestamps       : "
    f"{final_null_purchase_timestamps}"
)

print(
    f"NULL Silver load timestamps    : "
    f"{final_null_silver_timestamps}"
)

print(
    f"Incorrect approval flags       : "
    f"{final_incorrect_approval_flags}"
)

print(
    f"Incorrect timeline flags       : "
    f"{final_incorrect_timeline_flags}"
)

print("=" * 75)

if (
    final_silver_order_count == bronze_order_count
    and final_null_order_ids == 0
    and final_duplicate_order_ids == 0
    and final_null_customer_ids == 0
    and final_null_statuses == 0
    and final_invalid_statuses == 0
    and final_null_purchase_timestamps == 0
    and final_null_silver_timestamps == 0
    and final_incorrect_approval_flags == 0
    and final_incorrect_timeline_flags == 0
):
    print(
        "STATUS: ALL SILVER ORDERS CHECKS PASSED"
    )
else:
    raise ValueError(
        "SILVER ORDERS QUALITY GATE FAILED."
    )

SILVER ORDERS QUALITY GATE
Bronze rows                    : 99,441
Silver rows                    : 99,441
NULL order IDs                 : 0
Duplicate order IDs            : 0
NULL customer IDs              : 0
NULL order statuses            : 0
Invalid order statuses         : 0
NULL purchase timestamps       : 0
NULL Silver load timestamps    : 0
Incorrect approval flags       : 0
Incorrect timeline flags       : 0
STATUS: ALL SILVER ORDERS CHECKS PASSED



## Step 23 — Validate Silver Orders Delta Table

Verify that `workspace.silver.orders` is correctly persisted as a
managed Delta table.

The table must exist and contain the expected Silver dataset.

In [0]:
%sql

DESCRIBE DETAIL workspace.silver.orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,7605a397-a047-4844-95f5-5b4d7c165c10,workspace.silver.orders,null,,2026-08-11T09:35:05.369Z,2026-08-11T09:35:08.000Z,List(),List(),1,5721477,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false



## Step 24 — Validate Silver Orders Delta History

Verify the Delta transaction history for the Silver Orders table.

This provides operational traceability for table creation and future
pipeline executions.

In [0]:
%sql

DESCRIBE HISTORY workspace.silver.orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-11T09:35:08.000Z,74950116427109,harshrajparmar0117@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(954525762634559),4e216cbb-4f73-4da0-a03e-b3fa5a458674,0811-092638-jqtigxms-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 5721477)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13



## Step 25 — Verify Silver Orders Table Registration

Confirm that the Silver Orders table is registered in the
`workspace.silver` schema and available for downstream workloads.

In [0]:
%sql

SHOW TABLES IN workspace.silver;

database,tableName,isTemporary
silver,category_translation,false
silver,customers,false
silver,geolocation,false
silver,order_items,false
silver,order_payments,false
silver,order_reviews,false
silver,orders,false
silver,products,false
silver,sellers,false



# Silver Orders — Execution Summary

## Execution Status

**Status:** `SUCCESS`

---

## Source

**Bronze Table:**

`workspace.bronze.orders`

---

## Silver Target

`workspace.silver.orders`

---

## Source Grain

One record represents one customer order.

**Business Key:**

`order_id`

---

## Silver Transformations

The Silver Orders layer performs the following transformations:

- Trims `order_id`.
- Trims `customer_id`.
- Standardizes `order_status` to lowercase.
- Converts order lifecycle fields from STRING to TIMESTAMP.
- Creates `approval_missing_flag`.
- Creates `timeline_issue_flag`.
- Adds `silver_load_timestamp`.
- Preserves all Bronze order records.

No order records are removed because of lifecycle anomalies.

---

## Data Quality Validation

The following validations were completed successfully:

| Validation | Result |
|------------|--------|
| Bronze schema validation | PASS |
| Bronze order ID NULL validation | PASS |
| Bronze order ID uniqueness | PASS |
| Bronze/Silver row-count reconciliation | PASS |
| Silver order ID NULL validation | PASS |
| Silver order ID uniqueness | PASS |
| Required customer ID validation | PASS |
| Required order status validation | PASS |
| Required purchase timestamp validation | PASS |
| Order status domain validation | PASS |
| Silver datatype validation | PASS |
| Silver load timestamp validation | PASS |
| Approval flag validation | PASS |
| Timeline flag validation | PASS |
| Bronze/Silver order ID reconciliation | PASS |
| Timestamp conversion validation | PASS |
| Final persisted Silver quality gate | PASS |
| Delta metadata validation | PASS |
| Delta history validation | PASS |
| Silver table registration | PASS |

---

## Lifecycle Quality Findings

The expected order lifecycle is:

`Purchase → Approval → Carrier Delivery → Customer Delivery`

The following source-data anomalies were identified:

- Approval before purchase: `0`
- Carrier delivery before approval: `1,359`
- Customer delivery before carrier delivery: `23`
- Total orders with timeline issues: `1,382`
- Delivered orders with missing approval: `14`

These records were **not deleted**.

Instead, the anomalies are represented through:

- `approval_missing_flag`
- `timeline_issue_flag`

This preserves the source business records while making the data-quality issues available for downstream analytics.

---

## Population Reconciliation

**Bronze Orders rows:** `99,441`

**Silver Orders rows:** `99,441`

**Bronze order IDs missing from Silver:** `0`

**Silver order IDs not present in Bronze:** `0`

Therefore, the Bronze and Silver order populations match exactly.

---

## Automation Readiness

Critical data-quality failures raise exceptions and are therefore
capable of failing the corresponding Databricks Workflow task.

Known source-data lifecycle anomalies are retained and flagged rather
than causing unnecessary record deletion.

The persisted Silver table is a managed Delta table registered as:

`workspace.silver.orders`

---

## Downstream Dependencies

The Silver Orders table will participate in downstream relationships
with:

- `workspace.silver.customers`
- `workspace.silver.order_items`
- `workspace.silver.order_payments`
- `workspace.silver.reviews`

Referential-integrity checks will be performed once the required
related Silver tables are available.

---

## Final Status

**SUCCESS — Silver Orders transformation, validation, persistence,
and technical quality checks completed successfully.**


## Step 27 — Final Silver Orders Preview

Review the final persisted Silver Orders records.

In [0]:
final_orders_preview_df = spark.table(
    "workspace.silver.orders"
)

display(
    final_orders_preview_df.limit(20)
)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_missing_flag,timeline_issue_flag,silver_load_timestamp
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,false,false,2026-08-11T09:35:05.833Z
